# Football scouting model: find players like your target

A FIFA career mode style scouting backend. Give it a reference player or a target profile and it returns the best matches from the **Premier League, La Liga and Bundesliga**, with a plain reason for each match and a radar chart.

**How it works in one line:** per-90 stats, compared only inside a position group, standardised, then ranked by distance (kNN style). No tuning, no deep learning.

## Data sources and attribution
All data is downloaded by this notebook from the original sources and cached in `raw/`. Nothing is simulated.

| Source | Used for | Terms / credit |
|---|---|---|
| [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets) (dcaribou), built from Transfermarkt | age, position, market value, contract expiry, club, appearances | Dataset published under CC0 by its maintainer. Underlying data is from [Transfermarkt](https://www.transfermarkt.com). Credit both. |
| [soccerdata](https://github.com/probberechts/soccerdata) (Pieter Robberechts) | scraper library for the two stats sources below | Apache 2.0 |
| [FBref](https://fbref.com) / Sports Reference (tried first) | advanced player stats if available | Check [their terms](https://www.sports-reference.com/termsofuse.html). Rate limited by the library. |
| [FBref snapshots on Kaggle](https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2024-2025) (hubertsidorowicz) | tackles, interceptions, clearances, blocks, recoveries, aerials, progressive passes, goalkeeper saves, 2024/25 and 2025/26 | Public Kaggle datasets of FBref tables, downloaded anonymously by the notebook. Check the dataset licence and FBref terms before redistributing. Only derived per-90 values are exported. |
| [Understat](https://understat.com) (fallback) | xG, xA, npxG, shots, key passes, xGChain, xGBuildup | Scraped politely and cached. Check the site terms before redistributing raw data. Only derived per-90 values are exported. |

Which stats source actually worked is decided at run time and printed below.

## 1. Config

In [ ]:
import os, re, sys, time, json, shutil, unicodedata, difflib, warnings, subprocess
import datetime as dt
from pathlib import Path
warnings.filterwarnings("ignore")

TODAY = dt.date.today()
# Start year of the most recent completed season (season 2025 = 2025/26). A season ends in June.
LAST_COMPLETED = TODAY.year - 1 if TODAY.month >= 7 else TODAY.year - 2

CONFIG = {
    # Transfermarkt datasets (public CSV.GZ links)
    "tm_base_url": "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/",
    "tm_files": ["players", "clubs", "player_valuations", "appearances", "games"],
    "tm_leagues": {"GB1": "Premier League", "ES1": "La Liga", "L1": "Bundesliga"},
    # soccerdata league ids -> display name
    "sd_leagues": {"ENG-Premier League": "Premier League",
                   "ESP-La Liga": "La Liga",
                   "GER-Bundesliga": "Bundesliga"},
    "stats_sources": ["fbref", "understat"],      # tried in this order
    # seasons: last 4 completed, as start years
    "seasons": list(range(LAST_COMPLETED - 3, LAST_COMPLETED + 1)),
    "min_minutes": 450,         # per player per season
    "pool_last_n": 2,           # profile = minutes-weighted pool of the latest N qualifying seasons
    "kaggle_api": "https://www.kaggle.com/api/v1/datasets/download/",
    # FBref snapshots published on Kaggle (season start year -> dataset slug)
    "kaggle_fbref": {2024: "hubertsidorowicz/football-players-stats-2024-2025",
                     2025: "hubertsidorowicz/football-players-stats-2025-2026"},
    "min_coverage": 0.7,        # a stat needs this share of players with data (rest are set to the group median)
    "min_features": 4,          # a position group needs this many usable features
    "n_clusters": 4,            # KMeans roles per group (fixed, no tuning)
    "use_pca": False,           # optional noise reduction before ranking
    "top_similar": 20,
    "max_json_mb": 10,
    # politeness
    "user_agent": "Mozilla/5.0 (compatible; scouting-model-notebook/1.0; educational, cached, low-rate)",
    "sleep_s": 2.0,
    "retries": 4,
}
SEASONS = CONFIG["seasons"]
LATEST = SEASONS[-1]
LEAGUE_NAMES = list(CONFIG["tm_leagues"].values())

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path(".")
RAW = WORK / "raw"
OUT = WORK / "export"
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
os.environ["SOCCERDATA_DIR"] = str(RAW / "soccerdata")
SRC_LOG = {"notes": []}     # collected for the final summary
print("Seasons:", [f"{s}/{str(s+1)[2:]}" for s in SEASONS], "| raw cache:", RAW.resolve())

In [ ]:
try:
    import soccerdata as sd
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soccerdata"], check=True)
    import soccerdata as sd
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": CONFIG["user_agent"]})

def download(url, dest):
    """Polite cached download: skip if present, retry with exponential backoff."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(CONFIG["retries"]):
        try:
            with SESSION.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                tmp = dest.with_name(dest.name + ".part")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
                tmp.rename(dest)
            time.sleep(CONFIG["sleep_s"])
            return dest
        except Exception as e:
            wait = CONFIG["sleep_s"] * 2 ** i
            print(f"  retry {i+1}/{CONFIG['retries']} for {dest.name}: {type(e).__name__} {str(e)[:80]} (sleep {wait:.0f}s)")
            time.sleep(wait)
    raise RuntimeError(f"Could not download {url}")

## 2. Source tests
Each source gets a small test first. We print the real columns and seasons, then adapt the feature set to what actually downloads.

### 2a. Transfermarkt datasets

In [ ]:
TM = RAW / "transfermarkt"
tm_paths = {}
for name in CONFIG["tm_files"]:
    tm_paths[name] = download(CONFIG["tm_base_url"] + f"{name}.csv.gz", TM / f"{name}.csv.gz")
    head = pd.read_csv(tm_paths[name], nrows=2)
    print(f"{name:18s} {tm_paths[name].stat().st_size/1e6:6.1f} MB  cols={list(head.columns)}")

def read_tm(name, cols=None, **kw):
    have = list(pd.read_csv(tm_paths[name], nrows=0).columns)
    use = [c for c in (cols or have) if c in have]
    return pd.read_csv(tm_paths[name], usecols=use, **kw)

app = read_tm("appearances", ["player_id", "game_id", "player_club_id", "date", "competition_id",
                              "minutes_played", "goals", "assists", "yellow_cards", "red_cards"])
app = app[app["competition_id"].isin(CONFIG["tm_leagues"])].copy()
app["date"] = pd.to_datetime(app["date"])
# Team results while the player is on the pitch (weighted by minutes played). These are the only
# defensive and goalkeeper signals in the working sources, so they carry team context.
games = read_tm("games", ["game_id", "home_club_id", "away_club_id", "home_club_goals", "away_club_goals"])
app = app.merge(games, on="game_id", how="left")
home = app["player_club_id"] == app["home_club_id"]
app["conceded"] = np.where(home, app["away_club_goals"], app["home_club_goals"])
app["scored"] = np.where(home, app["home_club_goals"], app["away_club_goals"])
w = app["minutes_played"].clip(upper=90) / 90
app["gc"] = app["conceded"] * w
app["team_gf"] = app["scored"] * w
app["clean_sheets"] = (app["conceded"] == 0) * w
app["min90"] = app["minutes_played"].clip(upper=90)
app["season"] = np.where(app["date"].dt.month >= 7, app["date"].dt.year, app["date"].dt.year - 1)
cov = app.groupby(["season", "competition_id"]).agg(players=("player_id", "nunique"), rows=("date", "size")).unstack()
print("\nTransfermarkt appearance coverage (3 leagues):")
display(cov.loc[cov.index >= min(SEASONS) - 1])
missing = [s for s in SEASONS if s not in set(app["season"])]
if missing:
    SRC_LOG["notes"].append(f"Transfermarkt appearances missing seasons {missing}")
    print("WARNING: missing seasons in Transfermarkt:", missing)

### 2b. Player performance stats
FBref is tried first (needs xG style columns to be useful). If it is blocked or has no advanced stats, Understat is used.

In [ ]:
import logging
logging.getLogger("soccerdata").setLevel(logging.WARNING)

def flat(cols):
    out = []
    for c in cols:
        parts = c if isinstance(c, tuple) else (c,)
        parts = [str(p) for p in parts if str(p) and not str(p).startswith("Unnamed")]
        out.append("_".join(parts).lower())
    return out

# canonical stat -> (fbref stat_type, regex on flattened column name)
FBREF_MAP = {
    "minutes":        ("standard",   r"playing time_min$"),
    "goals":          ("standard",   r"performance_gls$"),
    "assists":        ("standard",   r"performance_ast$"),
    "xg":             ("standard",   r"expected_xg$"),
    "npxg":           ("standard",   r"expected_npxg$"),
    "xag":            ("standard",   r"expected_xag$"),
    "prog_carries":   ("standard",   r"progression_prgc$"),
    "prog_passes":    ("standard",   r"progression_prgp$"),
    "shots":          ("shooting",   r"standard_sh$"),
    "sot":            ("shooting",   r"standard_sot$"),
    "key_passes":     ("passing",    r"^kp$"),
    "passes_cmp":     ("passing",    r"total_cmp$"),
    "ppa":            ("passing",    r"^ppa$"),
    "tackles":        ("defense",    r"tackles_tkl$"),
    "interceptions":  ("defense",    r"^int$"),
    "blocks":         ("defense",    r"blocks_blocks$"),
    "clearances":     ("defense",    r"^clr$"),
    "touches":        ("possession", r"touches_touches$"),
    "dribbles_succ":  ("possession", r"take-ons_succ$"),
}
KEYS = ["league", "season", "team", "player"]

def season_start(s):
    """soccerdata season labels vary ('2223', '2022-2023', '22-23'); return the start year."""
    s = str(s)
    m = re.match(r"^(\d{4})\D+(\d{2,4})$", s)
    if m: return int(m.group(1))
    m = re.match(r"^(\d{2})\D?(\d{2})$", s)
    if m: return 2000 + int(m.group(1))
    raise ValueError(f"Unrecognised season label {s!r}")

def season_labels(seasons):
    return [f"{s}-{s+1}" for s in seasons]

def fbref_table(stat_type, seasons, leagues):
    fb = sd.FBref(leagues, season_labels(seasons), data_dir=RAW / "soccerdata" / "fbref")
    return fb.read_player_season_stats(stat_type).reset_index()

def probe_fbref():
    """Small test: one league-season, standard table. Useful only if it has xG style columns."""
    try:
        t = fbref_table("standard", [LATEST], ["ENG-Premier League"])
        cols = flat(t.columns)
        print("FBref reachable. standard columns:", cols[:30])
        has_xg = any(re.search(r"xg$", c) for c in cols)
        print("FBref has xG columns:", has_xg)
        return has_xg
    except Exception as e:
        print("FBref failed:", type(e).__name__, str(e)[:200])
        return False

def probe_understat():
    try:
        u = sd.Understat(["ENG-Premier League"], season_labels([LATEST]), data_dir=RAW / "soccerdata" / "understat")
        t = u.read_player_season_stats().reset_index()
        print("Understat reachable. columns:", list(t.columns))
        print("Understat seasons in test:", sorted(t["season"].unique()))
        return True
    except Exception as e:
        print("Understat failed:", type(e).__name__, str(e)[:200])
        return False

STATS_CACHE = RAW / "player_stats_clean.csv.gz"
STATS_SRC_FILE = RAW / "player_stats_source.txt"
STATS_SOURCE = None
if STATS_CACHE.exists() and STATS_SRC_FILE.exists():
    STATS_SOURCE = STATS_SRC_FILE.read_text().strip()
    print("Using cached cleaned stats from source:", STATS_SOURCE)
else:
    for src in CONFIG["stats_sources"]:
        print(f"--- testing {src} ---")
        ok = probe_fbref() if src == "fbref" else probe_understat()
        if ok:
            STATS_SOURCE = src
            break
        SRC_LOG["notes"].append(f"{src} unavailable or without advanced stats, fell back to next source")
    if STATS_SOURCE is None:
        raise RuntimeError("Every stats source failed. Cannot build the model without performance stats.")
print("Stats source selected:", STATS_SOURCE)

## 3. Load and clean stats
Standardise columns to one schema (`league, season, team, player, minutes` plus counting stats), keep the 3 leagues and chosen seasons.

In [ ]:
def load_fbref():
    frames = {}
    for stat_type in sorted({v[0] for v in FBREF_MAP.values()}):
        try:
            t = fbref_table(stat_type, SEASONS, list(CONFIG["sd_leagues"]))
        except Exception as e:
            print(f"  FBref table {stat_type} failed: {type(e).__name__}")
            SRC_LOG["notes"].append(f"FBref table {stat_type} unavailable")
            continue
        t.columns = flat(t.columns)
        part = t[KEYS].copy()
        for canon, (st, rx) in FBREF_MAP.items():
            if st != stat_type: continue
            hit = [c for c in t.columns if re.search(rx, c)]
            if hit: part[canon] = pd.to_numeric(t[hit[0]], errors="coerce")
        frames[stat_type] = part.drop_duplicates(KEYS)
    df = frames["standard"]
    for k, f in frames.items():
        if k != "standard":
            df = df.merge(f, on=KEYS, how="left")
    return df

def load_understat():
    u = sd.Understat(list(CONFIG["sd_leagues"]), season_labels(SEASONS), data_dir=RAW / "soccerdata" / "understat")
    t = u.read_player_season_stats().reset_index()
    t = t.rename(columns={"np_xg": "npxg", "xa": "xag"})
    keep = KEYS + ["minutes", "goals", "assists", "xg", "npxg", "xag", "shots", "key_passes", "xg_chain", "xg_buildup"]
    return t[[c for c in keep if c in t.columns]]

if STATS_CACHE.exists() and STATS_SRC_FILE.exists():
    stats = pd.read_csv(STATS_CACHE)
else:
    stats = load_fbref() if STATS_SOURCE == "fbref" else load_understat()
    stats["season"] = stats["season"].map(season_start)
    stats.to_csv(STATS_CACHE, index=False)
    STATS_SRC_FILE.write_text(STATS_SOURCE)

stats["league"] = stats["league"].map(CONFIG["sd_leagues"]).fillna(stats["league"])
stats = stats[stats["league"].isin(LEAGUE_NAMES) & stats["season"].isin(SEASONS)].copy()
stats["minutes"] = pd.to_numeric(stats["minutes"], errors="coerce")
stats = stats.dropna(subset=["minutes", "player"]).reset_index(drop=True)
SRC_LOG["stats_source"] = STATS_SOURCE
print(f"Stats rows: {len(stats):,} | columns: {list(stats.columns)}")
print("Seasons present:", sorted(stats['season'].unique()), "| leagues:", sorted(stats['league'].unique()))
display(stats.groupby(["season", "league"]).size().unstack())

### Extra FBref stats from Kaggle snapshots
FBref blocks scripts, and in 2025/26 it removed most advanced tables. Two public Kaggle datasets hold FBref tables: 2024/25 has the full set (tackles, clearances, blocks, recoveries, aerials, progressive actions, PSxG) and 2025/26 has only basics plus goalkeeper stats, interceptions and fouls. These are added to the Understat rows after name matching. A stat missing for a season is left empty, and each stat is rated per 90 using only the minutes of seasons where it exists.

In [ ]:
import zipfile, io
FBK_MAP = {"fb_min": "Min", "tackles": "Tkl", "tackles_won": "TklW", "interceptions": "Int", "blocks": "Blocks_stats_defense",
           "clearances": "Clr", "recoveries": "Recov", "aerials_won": "Won", "prog_passes": "PrgP", "prog_carries": "PrgC",
           "passes_cmp": "Cmp", "touches": "Touches", "dribbles_succ": "Succ", "ppa": "PPA", "sca": "SCA", "fouls": "Fls",
           "crosses": "Crs", "saves": "Saves", "sota": "SoTA", "psxg_diff": "PSxG+/-"}
FBK_LEAGUES = {"eng Premier League": "Premier League", "es La Liga": "La Liga", "de Bundesliga": "Bundesliga"}

def load_fbref_kaggle():
    frames = []
    for season, slug in CONFIG["kaggle_fbref"].items():
        if season not in SEASONS: continue
        try:
            z = download(CONFIG["kaggle_api"] + slug, RAW / "kaggle" / (slug.replace("/", "__") + ".zip"))
            with zipfile.ZipFile(z) as zf:
                name = [n for n in zf.namelist() if n.startswith("players_data-")][0]
                raw = pd.read_csv(zf.open(name))
        except Exception as e:
            print(f"  Kaggle FBref {season} unavailable: {type(e).__name__} {str(e)[:100]}")
            SRC_LOG["notes"].append(f"Kaggle FBref snapshot {season} unavailable")
            continue
        raw = raw[raw["Comp"].isin(FBK_LEAGUES)]
        d = pd.DataFrame({"league": raw["Comp"].map(FBK_LEAGUES), "season": season, "team": raw["Squad"], "player": raw["Player"]})
        for canon, col in FBK_MAP.items():
            if col in raw.columns: d[canon] = pd.to_numeric(raw[col], errors="coerce").values
        print(f"  {slug}: {len(d)} rows, stats present: {[c for c in d.columns if c in FBK_MAP and d[c].notna().mean() > 0.5 and c != 'fb_min']}")
        frames.append(d)
    if not frames: return pd.DataFrame()
    d = pd.concat(frames, ignore_index=True)
    drop = [c for c in d.columns if c in stats.columns and c not in KEYS]       # keep the primary source's version
    d = d.drop(columns=[c for c in drop if c != "fb_min"])
    return d.dropna(subset=["fb_min"])

fb = load_fbref_kaggle()
SRC_LOG["kaggle_fbref_rows"] = int(len(fb))
print("FBref (Kaggle) rows:", len(fb))

## 4. Merge with Transfermarkt
Names are normalised (accents stripped, lowercase, punctuation removed). Matching runs in three passes inside the same league and season:
1. exact normalised name,
2. same words in any order (handles "Son Heung-min" vs "Heung-min Son"),
3. close fuzzy name (similarity 0.88 or higher) with a similar club name.

When several candidates remain, the closest club name wins.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    for a, b in {"\u00f8": "o", "\u00df": "ss", "\u00e6": "ae", "\u0111": "d", "\u0142": "l", "\u0131": "i"}.items():
        s = s.replace(a, b)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def tokkey(s):
    return " ".join(sorted(norm(s).split()))

players = read_tm("players", ["player_id", "name", "date_of_birth", "position", "sub_position",
                              "contract_expiration_date", "country_of_citizenship", "foot", "height_in_cm"])
clubs = read_tm("clubs", ["club_id", "name"]).rename(columns={"club_id": "player_club_id", "name": "tm_club"})
vals = read_tm("player_valuations", ["player_id", "date", "market_value_in_eur"])
vals["date"] = pd.to_datetime(vals["date"])
latest_val = vals.sort_values("date").groupby("player_id").tail(1)[["player_id", "market_value_in_eur", "date"]]
latest_val = latest_val.rename(columns={"date": "value_date"})

# One Transfermarkt row per player, season, league (club with the most minutes), for name matching
tm_ps = (app.groupby(["player_id", "season", "competition_id", "player_club_id"], as_index=False)
            .agg(tm_minutes=("minutes_played", "sum"), tm_min=("min90", "sum"), yellow_cards=("yellow_cards", "sum"),
                  red_cards=("red_cards", "sum"), gc=("gc", "sum"), clean_sheets=("clean_sheets", "sum"),
                  team_gf=("team_gf", "sum")))
tm_ps["league"] = tm_ps["competition_id"].map(CONFIG["tm_leagues"])
tm_ps = tm_ps.merge(clubs, on="player_club_id", how="left").merge(players[["player_id", "name"]], on="player_id", how="left")
tm_ps = tm_ps.dropna(subset=["name"])
tm_ps = tm_ps.sort_values("tm_minutes", ascending=False).drop_duplicates(["player_id", "season", "league"])
tm_ps["key1"] = tm_ps["name"].map(norm); tm_ps["key2"] = tm_ps["name"].map(tokkey)
tm_ps["club_n"] = tm_ps["tm_club"].fillna("").map(norm)

def club_sim(a, b):
    a, b = a.replace(" fc", ""), b.replace(" fc", "")
    return difflib.SequenceMatcher(None, a, b).ratio() if not (a in b or b in a) else 1.0

def match_tm(df):
    st = df.copy().reset_index(drop=True)
    st["sid"] = np.arange(len(st))
    st["key1"] = st["player"].map(norm); st["key2"] = st["player"].map(tokkey); st["club_n"] = st["team"].map(norm)
    found, remaining = [], st
    for how, key in [("exact", "key1"), ("tokens", "key2")]:
        pool = tm_ps[[key, "league", "season", "player_id", "club_n", "tm_minutes"]].rename(columns={"club_n": "tm_club_n"})
        cand = remaining.merge(pool, on=[key, "league", "season"])
        cand["club_sim"] = [club_sim(a, b) for a, b in zip(cand["club_n"], cand["tm_club_n"])]
        cand = cand.sort_values(["sid", "club_sim", "tm_minutes"], ascending=[True, False, False]).drop_duplicates("sid")
        cand["match_type"] = how
        found.append(cand[["sid", "player_id", "match_type"]])
        remaining = remaining[~remaining["sid"].isin(cand["sid"])]
    rows = []
    for (lg, se), grp in remaining.groupby(["league", "season"]):
        pool = tm_ps[(tm_ps["league"] == lg) & (tm_ps["season"] == se)].reset_index(drop=True)
        names = pool["key2"].tolist()
        for r in grp.itertuples():
            best = None
            for nm in difflib.get_close_matches(r.key2, names, n=3, cutoff=0.88):
                for j in np.where(pool["key2"].values == nm)[0]:
                    cs = club_sim(r.club_n, pool.loc[j, "club_n"])
                    if cs >= 0.5 and (best is None or cs > best[1]):
                        best = (pool.loc[j, "player_id"], cs)
            if best: rows.append({"sid": r.sid, "player_id": best[0], "match_type": "fuzzy"})
    found.append(pd.DataFrame(rows, columns=["sid", "player_id", "match_type"]))
    st = st.merge(pd.concat(found, ignore_index=True), on="sid", how="left")
    st["matched"] = st["player_id"].notna()
    return st

def report_match(st, label, min_col):
    elig = st[st[min_col] >= CONFIG["min_minutes"]]
    rate = elig["matched"].mean()
    print(f"[{label}] match rate, all rows: {st['matched'].mean():.1%} | rows with >= {CONFIG['min_minutes']} min: {rate:.1%}")
    print(elig["match_type"].fillna("unmatched").value_counts().to_string())
    display(elig.groupby("season")["matched"].mean().round(3).to_frame("match_rate").T)
    print("Sample of unmatched players (>= min minutes):")
    display(elig[~elig["matched"]][["player", "team", "league", "season", min_col]]
            .sort_values(min_col, ascending=False).head(15).reset_index(drop=True))
    return rate

st = match_tm(stats)
rate_elig = report_match(st, f"stats: {STATS_SOURCE}", "minutes")
SRC_LOG["match_rate"] = round(float(rate_elig), 4)
if len(fb):
    fbm = match_tm(fb)
    SRC_LOG["match_rate_fbref_kaggle"] = round(float(report_match(fbm, "FBref via Kaggle", "fb_min")), 4)
else:
    fbm = fb

## 5. Feature engineering
- Aggregate to one row per player and season (a mid-season club move inside the leagues is summed).
- Drop player seasons under the minutes threshold.
- Build each profile from the latest qualifying seasons (minutes-weighted), then convert counts to **per 90**.
- Only players who qualify in the latest season are kept, so recommendations are current.
- Position groups come from the Transfermarkt sub-position.

In [ ]:
m = st[st["matched"]].copy()
m["player_id"] = m["player_id"].astype(int)
count_cols = [c for c in stats.columns if c not in KEYS + ["minutes"]]

m = m.sort_values("minutes", ascending=False)
meta = m.groupby(["player_id", "season"]).agg(club=("team", "first"), league=("league", "first"))
sums = m.groupby(["player_id", "season"])[["minutes"] + count_cols].sum(min_count=1)
ps = sums.join(meta).reset_index()
fb_cols = []
if len(fbm):
    fb_cols = [c for c in fbm.columns if c in FBK_MAP and c != "fb_min"]
    fbx = (fbm[fbm["matched"]].assign(player_id=lambda d: d["player_id"].astype(int))
           .groupby(["player_id", "season"])[["fb_min"] + fb_cols].sum(min_count=1).reset_index())
    ps = ps.merge(fbx, on=["player_id", "season"], how="left")
else:
    ps["fb_min"] = np.nan
TM_COLS = ["yellow_cards", "red_cards", "gc", "clean_sheets", "team_gf"]   # rates use Transfermarkt minutes
tmx = tm_ps.groupby(["player_id", "season"])[["tm_min"] + TM_COLS].sum().reset_index()
ps = ps.merge(tmx, on=["player_id", "season"], how="left")
ps = ps[ps["minutes"] >= CONFIG["min_minutes"]]
print(f"Player seasons kept (>= {CONFIG['min_minutes']} min): {len(ps):,}")

# pool the latest N qualifying seasons per player
ps = ps.sort_values(["player_id", "season"], ascending=[True, False])
ps["rank"] = ps.groupby("player_id").cumcount()
latest_row = ps[ps["rank"] == 0]
active_ids = set(latest_row.loc[latest_row["season"] == LATEST, "player_id"])
pool_rows = ps[(ps["rank"] < CONFIG["pool_last_n"]) & ps["player_id"].isin(active_ids)]
agg = pool_rows.groupby("player_id")[["minutes", "tm_min", "fb_min"]].sum(min_count=1)
MIN_OF = {**{c: "minutes" for c in count_cols}, **{c: "tm_min" for c in TM_COLS}, **{c: "fb_min" for c in fb_cols}}
for c, mc in MIN_OF.items():
    # rate over only the seasons where this stat exists
    ok = pool_rows[c].notna() & (pool_rows[mc] > 0)
    num = pool_rows[c].where(ok).groupby(pool_rows["player_id"]).sum(min_count=1)
    den = pool_rows[mc].where(ok).groupby(pool_rows["player_id"]).sum(min_count=1)
    agg[c + "_p90"] = num / den * 90
if {"saves_p90", "sota_p90"} <= set(agg.columns):
    agg["save_pct_p90"] = agg["saves_p90"] / agg["sota_p90"].where(agg["sota_p90"] > 0) * 100   # save %, not a per-90 value
agg = agg.join(latest_row.set_index("player_id")[["club", "league"]])

POS_MAP = {"Goalkeeper": "GK", "Centre-Back": "CB", "Left-Back": "FB", "Right-Back": "FB",
           "Defensive Midfield": "DM/CM", "Central Midfield": "DM/CM",
           "Attacking Midfield": "AM/W", "Left Midfield": "AM/W", "Right Midfield": "AM/W",
           "Left Winger": "AM/W", "Right Winger": "AM/W", "Centre-Forward": "ST", "Second Striker": "ST"}
BROAD = {"Goalkeeper": "GK", "Defender": "CB", "Midfield": "DM/CM", "Attack": "ST"}
pl = players.set_index("player_id")
P = agg.join(pl).join(latest_val.set_index("player_id"))
P["group"] = P["sub_position"].map(POS_MAP).fillna(P["position"].map(BROAD))
P["height_cm_p90"] = P["height_in_cm"].where(P["height_in_cm"] > 100)
P["age"] = ((pd.Timestamp(TODAY) - pd.to_datetime(P["date_of_birth"])).dt.days / 365.25).round(1)
P["value_eur"] = P["market_value_in_eur"]
P = P.dropna(subset=["group"]).reset_index().rename(columns={"index": "player_id"})
print(f"Active players in latest season ({LATEST}/{str(LATEST+1)[2:]}): {len(P):,}")
display(P.groupby(["group", "league"]).size().unstack(fill_value=0))
print("Market value coverage:", f"{P['value_eur'].notna().mean():.1%}", "| latest valuation date range:",
      P['value_date'].min().date(), "to", P['value_date'].max().date())

### Feature sets per position group
Each group lists candidate stats. A stat is used only if it exists in the downloaded data with at least 70% coverage in that group (players missing a stat get the group median for ranking and an empty value in the export).

**Defenders, midfielders and goalkeepers.** Understat has no defensive actions, so these groups use the FBref Kaggle snapshots (tackles, interceptions, clearances, blocks, recoveries, aerial wins, progressive passes, goalkeeper saves, shots faced, save %, PSxG +/-) plus Transfermarkt-derived signals (cards, height, and for goalkeepers team goals conceded and clean sheets while on the pitch). Advanced FBref stats exist only for 2024/25, so a player who joined in 2025/26 has them empty. A group is labelled `limited` if none of its individual defensive or goalkeeper stats loaded. Groups with fewer than the minimum number of features are skipped and reported.

In [ ]:
CANDIDATES = {
    "ST":    ["npxg", "shots", "sot", "goals", "xag", "key_passes", "xg_chain", "xg_buildup", "dribbles_succ", "touches", "sca", "aerials_won"],
    "AM/W":  ["npxg", "shots", "xag", "key_passes", "assists", "xg_chain", "xg_buildup", "dribbles_succ", "prog_carries", "ppa", "sca", "crosses"],
    "DM/CM": ["xg_buildup", "xg_chain", "key_passes", "xag", "npxg", "prog_passes", "prog_carries", "passes_cmp", "tackles", "interceptions",
              "recoveries", "aerials_won", "fouls", "yellow_cards", "height_cm"],
    "FB":    ["xg_buildup", "xg_chain", "key_passes", "xag", "prog_carries", "prog_passes", "crosses", "dribbles_succ", "tackles", "interceptions",
              "recoveries", "aerials_won", "fouls", "yellow_cards"],
    "CB":    ["tackles", "interceptions", "clearances", "blocks", "aerials_won", "recoveries", "prog_passes", "passes_cmp", "xg_buildup",
              "fouls", "yellow_cards", "height_cm"],
    "GK":    ["save_pct", "saves", "sota", "psxg_diff", "gc", "clean_sheets", "height_cm"],
}
DEFENSIVE = {"tackles", "interceptions", "clearances", "blocks", "aerials_won", "recoveries", "save_pct", "saves"}
LABEL = {"npxg": "npxG", "xag": "xA", "xg": "xG", "xg_chain": "xG chain", "xg_buildup": "build-up", "key_passes": "key passes",
         "shots": "shots", "sot": "shots on target", "goals": "goals", "assists": "assists", "dribbles_succ": "dribbles",
         "touches": "touches", "prog_carries": "prog. carries", "prog_passes": "prog. passes", "ppa": "passes into box",
         "tackles": "tackles", "interceptions": "interceptions", "clearances": "clearances", "blocks": "blocks",
         "passes_cmp": "passing volume", "yellow_cards": "yellow cards", "red_cards": "red cards",
         "tackles_won": "tackles won", "recoveries": "recoveries", "aerials_won": "aerial wins", "sca": "shot-creating actions",
         "fouls": "fouls", "crosses": "crosses", "saves": "saves", "sota": "shots faced", "save_pct": "save %", "psxg_diff": "PSxG +/-",
         "gc": "goals conceded on pitch", "clean_sheets": "clean sheets", "team_gf": "team goals for", "height_cm": "height"}

FEATS, GROUP_NOTES = {}, {}
for g, cands in CANDIDATES.items():
    sub = P[P["group"] == g]
    use = [c for c in cands if c + "_p90" in P.columns and len(sub) and sub[c + "_p90"].notna().mean() >= CONFIG["min_coverage"]
           and sub[c + "_p90"].std() > 0]
    if len(use) >= CONFIG["min_features"]:
        FEATS[g] = use
        GROUP_NOTES[g] = "ok" if (g in ("ST", "AM/W") or set(use) & DEFENSIVE) else "limited (no individual defensive stats in this source)"
    else:
        GROUP_NOTES[g] = f"skipped ({len(use)} usable features, thin data)"
for g in CANDIDATES:
    print(f"{g:6s} players={int((P['group']==g).sum()):4d}  {GROUP_NOTES[g]:50s} {FEATS.get(g, '')}")
SRC_LOG["group_notes"] = GROUP_NOTES

## 6. Scale, model and roles
- Standardise features **within** each position group (z-scores, clipped at 3.5).
- Similarity is the RMS gap in z-scores mapped to 0-1: `similarity = 1 / (1 + rms_gap)`. A score of 0.5 means players differ by one standard deviation per stat on average.
- KMeans (fixed k, no tuning) names roles from each cluster's strongest features.
- PCA is available behind `CONFIG["use_pca"]`. It is off by default so every stat stays explainable.

In [ ]:
MODEL = {}
for g, feats in FEATS.items():
    sub = P[P["group"] == g].copy().reset_index(drop=True)
    cols = [f + "_p90" for f in feats]
    X = sub[cols].fillna(sub[cols].median())
    sc = StandardScaler().fit(X)
    Z = np.clip(sc.transform(X), -3.5, 3.5)
    k = max(2, min(CONFIG["n_clusters"], len(sub) // 25))
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    names = {}
    for c in range(k):
        cen = km.cluster_centers_[c]
        top = [feats[i] for i in np.argsort(-cen)[:2] if cen[i] > 0.3]
        base = " + ".join(LABEL.get(t, t) for t in top) if top else "balanced"
        n = base; j = 2
        while n in names.values():
            n, j = f"{base} ({j})", j + 1
        names[c] = n
    sub["role"] = [f"{g}: {names[c]}" for c in km.labels_]
    pct = sub[cols].rank(pct=True).mul(100).round(0)
    sub["mean_z"] = Z.mean(axis=1)
    sub["pid_ix"] = np.arange(len(sub))
    Zr = PCA(n_components=0.95, random_state=0).fit_transform(Z) if CONFIG["use_pca"] else Z
    MODEL[g] = dict(df=sub, feats=feats, cols=cols, scaler=sc, Z=Z, Zr=Zr, pct=pct)
    print(f"{g}: {len(sub)} players, {len(feats)} features, roles ->", sorted(set(sub['role'])))
INDEX = {int(pid): (g, i) for g, mm in MODEL.items() for i, pid in enumerate(mm["df"]["player_id"])}
print("Players in model:", len(INDEX))

## 7. Query function and explainability
`recommend()` takes a reference player or a raw per-90 `target_profile` (for example `{"npxg": 0.5, "xag": 0.3}`), and filters by age, value and league. It adds the top 3 stats where each candidate is closest to the reference, and a value-for-similarity score (`similarity / log10(value in millions + 10)`, higher is better value).

In [ ]:
def resolve(name):
    """Find a player by name (accent insensitive). Returns player_id or raises with suggestions."""
    q = norm(name)
    pn = P["name"].map(norm)
    hit = P[pn == q]
    if hit.empty:
        toks = q.split()
        hit = P[pn.map(lambda s: all(t in s.split() for t in toks))]
    if hit.empty:
        sug = difflib.get_close_matches(q, pn.tolist(), n=5, cutoff=0.6)
        raise KeyError(f"No active player matching {name!r}. Close names: {[P.loc[pn == s, 'name'].iloc[0] for s in sug]}")
    return int(hit.sort_values("minutes", ascending=False)["player_id"].iloc[0])

def explain(g, i_ref, i_cand, n=3):
    """Top n stats where the candidate is closest to the reference (preferring stats that define the reference)."""
    mm = MODEL[g]
    d = np.abs(mm["Z"][i_ref] - mm["Z"][i_cand])
    rel = np.abs(mm["Z"][i_ref]) >= 0.5
    order = sorted(range(len(d)), key=lambda j: (not rel[j], d[j]))[:n]
    return [(mm["feats"][j], float(mm["df"].loc[i_ref, mm["cols"][j]]), float(mm["df"].loc[i_cand, mm["cols"][j]])) for j in order]

def recommend(reference_player=None, position_group=None, max_age=None, max_value=None, leagues=None,
              top_n=10, target_profile=None, sort_by="similarity"):
    """Ranked look-alikes. max_value is in EUR. sort_by is 'similarity' or 'value'."""
    ref_i = None
    if reference_player is not None:
        pid = resolve(reference_player) if isinstance(reference_player, str) else int(reference_player)
        g, ref_i = INDEX[pid]
        if position_group and position_group != g:
            print(f"Note: {reference_player} is a {g}, ignoring position_group={position_group}")
    else:
        g = position_group
        if g not in MODEL:
            raise ValueError(f"position_group must be one of {list(MODEL)}")
    mm = MODEL[g]; df = mm["df"]; feats = mm["feats"]; Z = mm["Z"]
    if ref_i is not None:
        sel = list(range(len(feats))); zt = Z[ref_i]
    elif target_profile:
        sel = [feats.index(k) for k in target_profile if k in feats]
        if not sel: raise ValueError(f"target_profile keys must be in {feats}")
        mu, sdv = mm["scaler"].mean_, mm["scaler"].scale_
        zt = np.zeros(len(feats))
        for k in target_profile:
            if k in feats:
                j = feats.index(k); zt[j] = np.clip((target_profile[k] - mu[j]) / sdv[j], -3.5, 3.5)
    else:
        sel = None
    if sel is not None:
        rms = np.sqrt(((Z[:, sel] - zt[sel]) ** 2).mean(axis=1))
        sim = 1 / (1 + rms)
    else:                                   # no reference: best overall producers in the group
        sim = df["mean_z"].rank(pct=True).values
    out = df.copy(); out["similarity"] = sim
    mask = pd.Series(True, index=out.index)
    if ref_i is not None: mask &= out.index != ref_i
    if max_age is not None: mask &= out["age"] <= max_age
    if max_value is not None: mask &= out["value_eur"].fillna(np.inf) <= max_value
    if leagues: mask &= out["league"].isin(leagues)
    out = out[mask]
    vm = out["value_eur"] / 1e6
    out["value_score"] = out["similarity"] / np.log10(vm + 10)
    out = out.sort_values("similarity" if sort_by == "similarity" else "value_score", ascending=False).head(top_n)
    if ref_i is not None:
        out["why"] = [", ".join(f"{LABEL.get(f, f)} ({a:.2f} vs {b:.2f})" for f, a, b in explain(g, ref_i, i)) for i in out.index]
    res = out[["name", "club", "league", "age", "role", "value_eur", "similarity", "value_score"]].copy()
    res["value_m_eur"] = (res.pop("value_eur") / 1e6).round(1)
    res["similarity"] = res["similarity"].round(3); res["value_score"] = res["value_score"].round(3)
    if ref_i is not None: res["why"] = out["why"]
    return res.reset_index(drop=True)

def radar(reference, candidate):
    """Radar chart of within-group percentiles (0-100) for two players."""
    a, b = resolve(reference), resolve(candidate)
    (ga, ia), (gb, ib) = INDEX[a], INDEX[b]
    assert ga == gb, "Players must be in the same position group"
    mm = MODEL[ga]; feats = mm["feats"]
    va, vb = mm["pct"].iloc[ia].values, mm["pct"].iloc[ib].values
    ang = np.linspace(0, 2 * np.pi, len(feats), endpoint=False).tolist(); ang += ang[:1]
    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
    for vals, nm, col in [(va, mm["df"].loc[ia, "name"], "#1f77b4"), (vb, mm["df"].loc[ib, "name"], "#d95f02")]:
        v = list(vals) + [vals[0]]
        ax.plot(ang, v, color=col, lw=2, label=nm); ax.fill(ang, v, color=col, alpha=0.15)
    ax.set_xticks(ang[:-1]); ax.set_xticklabels([LABEL.get(f, f) for f in feats], fontsize=8)
    ax.set_ylim(0, 100); ax.set_yticks([25, 50, 75]); ax.set_yticklabels(["25", "50", "75"], fontsize=7)
    ax.set_title(f"{ga}: percentile within group (per 90)", fontsize=10, pad=18)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout(); plt.show()

## 8. Validation
Sanity check with well-known players. The first five found among the candidates are used. Judge the lists by eye: do the neighbours play a similar role?

In [ ]:
WISHLIST = ["Erling Haaland", "Mohamed Salah", "Rodri", "Pedri", "Virgil van Dijk", "Harry Kane", "Declan Rice",
            "Florian Wirtz", "Bukayo Saka", "Jamal Musiala", "Jude Bellingham", "Lamine Yamal", "Joshua Kimmich"]
checked = []
for nm in WISHLIST:
    if len(checked) == 5: break
    try:
        pid = resolve(nm)
    except KeyError:
        continue
    if P.set_index("player_id").loc[pid, "group"] not in MODEL:
        continue
    checked.append(nm)
    print(f"\n=== Top 5 like {nm} ({P.set_index('player_id').loc[pid, 'group']}) ===")
    display(recommend(nm, top_n=5))
print("\nPlayers checked:", checked)
SRC_LOG["validated"] = checked

In [ ]:
# Example queries: undervalued young alternatives, and a profile-based search
if checked:
    ref = checked[0]
    print(f"Under 24 and under EUR 40m, like {ref}:")
    display(recommend(ref, max_age=24, max_value=40e6, top_n=5, sort_by="value"))
    other = [c for c in checked if INDEX[resolve(c)][0] == INDEX[resolve(ref)][0] and c != ref]
    top1 = recommend(ref, top_n=1)["name"].iloc[0]
    radar(ref, top1)
if "ST" in MODEL and "npxg" in MODEL["ST"]["feats"]:
    print("Profile search: strikers with npxG 0.5/90 and 3 shots/90")
    display(recommend(position_group="ST", target_profile={"npxg": 0.5, "shots": 3.0}, top_n=5))

## 9. Export for the static web UI
- `players.json`: one record per player (latest season) with per-90 stats, percentiles, role, value, age.
- `similar_players.json`: top 20 look-alikes per player (same position group) with scores and the top 3 matching stats.

The site does all filtering in JavaScript, so no backend is needed.

In [ ]:
def clean(v):
    if v is None: return None
    if isinstance(v, (np.floating, float)): return None if np.isnan(v) else round(float(v), 3)
    if isinstance(v, (np.integer,)): return int(v)
    return v

players_out, similar_out = [], {}
for g, mm in MODEL.items():
    df, feats, cols = mm["df"], mm["feats"], mm["cols"]
    D = np.sqrt(((mm["Zr"][:, None, :] - mm["Zr"][None, :, :]) ** 2).mean(axis=2))
    S = 1 / (1 + D)
    for i, r in df.iterrows():
        pid = int(r["player_id"])
        players_out.append({
            "id": pid, "name": r["name"], "club": r["club"], "league": r["league"],
            "age": clean(r["age"]), "position_group": g, "position": r["sub_position"],
            "market_value_eur": None if pd.isna(r["value_eur"]) else int(r["value_eur"]),
            "contract_expiry": None if pd.isna(r["contract_expiration_date"]) else str(r["contract_expiration_date"])[:10],
            "minutes": int(r["minutes"]), "role": r["role"], "profile_quality": GROUP_NOTES[g].split(" ")[0],
            "stats": {f: clean(r[c]) for f, c in zip(feats, cols)},
            "pct": {f: clean(mm["pct"].loc[i, c]) for f, c in zip(feats, cols)},
        })
        order = [j for j in np.argsort(-S[i]) if j != i][:CONFIG["top_similar"]]
        similar_out[str(pid)] = [{"id": int(df.loc[j, "player_id"]), "score": round(float(S[i, j]), 3),
                                  "top": [f for f, _, _ in explain(g, i, j)]} for j in order]

for fname, obj in [("players.json", players_out), ("similar_players.json", similar_out)]:
    path = OUT / fname
    path.write_text(json.dumps(obj, separators=(",", ":"), allow_nan=False, ensure_ascii=False), encoding="utf-8")
    mb = path.stat().st_size / 1e6
    print(f"{fname}: {mb:.2f} MB")
    assert mb < CONFIG["max_json_mb"], f"{fname} exceeds {CONFIG['max_json_mb']} MB"

repo_export = Path("..") / "export"       # when run from notebooks/ inside the repo clone
if not ON_KAGGLE and repo_export.exists():
    for fname in ["players.json", "similar_players.json"]:
        shutil.copy(OUT / fname, repo_export / fname)
    print("Copied to", repo_export.resolve())
else:
    print("Download", [str(OUT / f) for f in ["players.json", "similar_players.json"]], "and put them in the repo export/ folder.")
print("Sample record:", json.dumps(players_out[0])[:400])

## 10. Limitations
- **Stats depth depends on the source.** Understat only covers attacking and chance-creation stats (xG, xA, shots, key passes, xG chain, xG buildup). If the Kaggle snapshots fail to download, defenders and midfielders are described only by build-up involvement, cards and height, and are flagged `limited` in the export.
- **Defensive stats are one season old.** FBref removed its advanced tables in 2025/26, so tackles, clearances, blocks and similar stats come from 2024/25 only. Goalkeeper saves and shots faced cover both seasons. Newly signed players lack the 2024/25 stats.
- **Name matching is heuristic.** The match rate and a sample of misses are printed above. Unmatched players are left out of the model.
- **Market value is the latest Transfermarkt valuation**, not the value at the time of the season. Age is as of the run date.
- **Per-90 rates over about 900 to 1,800 minutes are noisy.** Pooling the last two qualifying seasons helps but does not remove it.
- **Style, not quality.** Distance on z-scores finds similar output levels and shapes. It does not model tactics, role context, or league strength differences.
- **No tuning or validation against ground truth.** The 5-player check is a sanity check, not an accuracy score.

In [ ]:
print("=== Run summary ===")
print("Stats source used:", SRC_LOG.get("stats_source"), "| FBref via Kaggle rows:", SRC_LOG.get("kaggle_fbref_rows"))
print("Match rate (>= min minutes):", SRC_LOG.get("match_rate"), "| FBref via Kaggle:", SRC_LOG.get("match_rate_fbref_kaggle"))
print("Players exported:", len(players_out))
print("Group notes:", SRC_LOG.get("group_notes"))
print("Notes:", SRC_LOG["notes"] or "none")